<a href="https://colab.research.google.com/github/EmilioUcelayLojo/AA3/blob/laboratorios_practicas/lab8/lab8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eirasf/GCED-AA3/blob/main/lab8/lab8.ipynb)

# Práctica 8: Modelos generativos

## Pre-requisitos

### Instalar paquetes

Si la práctica requiere algún paquete de Python, habrá que incluir una celda en la que se instalen. Si usamos un paquete que se ha utilizado en prácticas anteriores, podríamos dar por supuesto que está instalado pero no cuesta nada satisfacer todas las dependencias en la propia práctica para reducir las dependencias entre ellas.

### NOTA: En <font color='red'>Google Colab</font> hay que instalar los paquetes EN CADA EJECUCIÓN

In [67]:
# Ejemplo de instalación de tensorflow 2.0
#%tensorflow_version 2.x
# !pip3 install tensorflow  # NECESARIO SOLO SI SE EJECUTA EN LOCAL
import tensorflow as tf

# Hacemos los imports que sean necesarios
import numpy as np

# Modelos generativos sobre MNIST

Lo primero que tenemos que hacer es cargar el dataset.

In [68]:
labeled_data = 0.01 # Vamos a usar el etiquetado de sólo el 1% de los datos
np.random.seed(42)

(x_train, y_train), (x_test, y_test), = tf.keras.datasets.mnist.load_data()

indices = np.arange(len(x_train))
np.random.shuffle(indices)
ntrain_data = int(labeled_data*len(x_train))
unlabeled_train = x_train[indices[ntrain_data:]]
x_train = x_train[indices[:ntrain_data]]
y_train = y_train[indices[:ntrain_data]]

In [69]:
# TODO: Haz el preprocesado que necesites aquí (si lo necesitas)
# Normalizar los valores de los píxeles al rango [0, 1]
x_train = x_train.astype(np.float32) / 255.0
unlabeled_train = unlabeled_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0

# Aplanar las imágenes para que sean vectores de características
x_train = x_train.reshape(x_train.shape[0], -1)
unlabeled_train = unlabeled_train.reshape(unlabeled_train.shape[0], -1)
x_test = x_test.reshape(x_test.shape[0], -1)

## Modelo generativo

Vamos a crear nuestro propio modelo generativo. En clase de teoría has visto muchas versiones distintas:

1. Mezcla de distribuciones de Gaussianas (GMM)
1. Mezcla de distribuciones multinomiales (Naive Bayes)
1. Modelos de Markov ocultos (HMM)

Tal y como se os apunta en teoría, los modelos generativos abordan un problema más general que la clasificación o regresión: aprenden cómo se estructuran y distribuyen los datos de entrada.

En nuestro caso, vamos a modelar los datos de entrada mediante el uso de **Autoencoders**.

# Autoencoders

El autoencoder es un tipo de red que se utiliza para aprender codificaciones eficientes de datos sin etiquetar (lo que se conoce como aprendizaje no supervisado). Es una red que tiene el mismo tamaño en la entrada como en la salida, puesto que el objetivo de la red es reconstruir la entrada con la menor pérdida posible.

Si lo que hacemos es reconstruir la entrada, ¿qué sentido tiene el usar la red? Habitualmente, **la red consta, a su mitad, de una capa con menos elementos que los datos de entrada**. Por tanto, al reconstruir los datos de la entrada a la salida, en esa capa tendremos una versión *comprimida* de la entrada, que contendrá la mayor parte de su información.

Por tanto, podemos dividir un autoencoder en 3 secciones diferentes, tal y como se ve en la siguiente figura:

![](https://drive.google.com/uc?export=view&id=1yxkKZV0J0YplQAGPGJxQ2Z80Ad6L94eu)

1. **Encoder:** es la parte inicial de la red, encargada de comprimir los datos de la entrada.
1. **Code:** es la salida del encoder, contiene la versión *comprimida* de los datos de entrada.
1. **Decoder:** se encarga de, partiendo de la salida del *Encoder*, reconstruir la red.

## Crea tu propio Autoencoder

El diseño del autoencoder es libre (capas densas, convolucionales, ...), puedes crearlo como quieras. **El único requisito es que tiene que mantener los nombres (y parámetros) de las funciones descritas abajo.**

In [76]:
# TODO: crea tu propio autoencoder

class MiAutoencoder:

    def __init__(self, input_shape):
        # Define el modelo
        input_layer = tf.keras.layers.Input(shape=input_shape)  # Define la capa de entrada
        x = tf.keras.layers.Dense(128, activation='relu')(input_layer)
        x = tf.keras.layers.Dense(64, activation='relu')(x)
        encoded = tf.keras.layers.Dense(32, activation='relu')(x)  # Capa densa con 32 neuronas y activación ReLU (code)

        # Define el decoder
        x = tf.keras.layers.Dense(64, activation='relu')(encoded)
        x = tf.keras.layers.Dense(128, activation='relu')(x)
        decoded = tf.keras.layers.Dense(input_shape[0], activation='sigmoid')(x)  # Capa de salida con activación sigmoid

        # Define el autoencoder
        self.autoencoder = tf.keras.models.Model(inputs=input_layer, outputs=decoded)

        # Define el encoder
        self.encoder = tf.keras.models.Model(inputs=input_layer, outputs=encoded)

        # Compila el modelo
        self.autoencoder.compile(optimizer='adam', loss='mse')  # Optimizador Adam y función de pérdida MSE

    def fit(self, X, y=None, sample_weight=None):
        # Entrena el modelo
        self.autoencoder.fit(X, X, epochs=20, batch_size=256)  # 20 épocas y tamaño de batch 256

    def get_encoded_data(self, X):
        # Devuelve la salida del encoder (code)
        return self.encoder.predict(X)  # Predice usando el encoder para obtener el code

    def __del__(self):
        # Elimina los modelos
        tf.keras.backend.clear_session()  # Libera la memoria en GPU

## Crea tu propio Clasificador

A continuación crearemos un clasificador que sea capaz de predecir la etiqueta pero no a partir de los datos originales, sino de la codificación `code` aprendida por el autoencoder. El diseño del clasificador es libre, pero recuerda que tiene que ser simple (máximo dos capas). **El único requisito es que tiene que mantener los nombres (y parámetros) de las funciones descritas abajo.**

In [77]:
# TODO: crea tu propio clasificador

class MiClasificador:

    def __init__(self):
        # Define el modelo
        self.model = tf.keras.models.Sequential([
            tf.keras.layers.Dense(64, activation='relu', input_shape=(32,)),  # Capa densa con 64 neuronas y activación ReLU
            tf.keras.layers.Dense(10, activation='softmax')  # Capa de salida con 10 neuronas (para las 10 clases de MNIST) y activación softmax
        ])
        # Compila el modelo
        self.model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])  # Optimizador Adam, función de pérdida sparse_categorical_crossentropy y métrica accuracy

    def fit(self, X, y, sample_weight=None):
        # Entrena el modelo
        self.model.fit(X, y, epochs=10, batch_size=32, sample_weight=sample_weight)  # 10 épocas, tamaño de batch 32 y pesos de muestra opcionales

    def predict(self, X):
        # Devuelve la clase ganadora
        return np.argmax(self.model.predict(X), axis=-1)  # Predice las probabilidades y devuelve el índice de la clase con mayor probabilidad

    def predict_proba(self, X):
        # Devuelve las probabilidades de cada clase
        return self.model.predict(X)  # Predice las probabilidades de cada clase

    def score(self, X, y):
        # Devuelve la precisión del modelo
        _, accuracy = self.model.evaluate(X, y, verbose=0)  # Evalúa el modelo y devuelve la pérdida y la precisión
        return accuracy  # Devuelve la precisión

    def __del__(self):
        # Elimina el modelo
        tf.keras.backend.clear_session()  # Libera la memoria en GPU

### Entrenamiendo del modelo sin supervisar

Primero de todo, a modo de comparación, crea un modelo (de capacidad similar a tu encoder+clasificador) que puedas entrenar supervisadamente con `x_train` e `y_train`. Anota su rendimiento.

In [79]:
# TODO - Crea un modelo, entrénalo con x_train e y_train y muestra su rendimiento en test.

# Crea un modelo de clasificación simple
modelo_simple = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(784,)),  # Capa de entrada con la forma de los datos
    tf.keras.layers.Dense(64, activation='relu'),  # Capa densa con 64 neuronas y activación ReLU
    tf.keras.layers.Dense(10, activation='softmax')  # Capa de salida con 10 neuronas (para las 10 clases de MNIST) y activación softmax
])


# Compila el modelo
modelo_simple.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Entrena el modelo con x_train e y_train
modelo_simple.fit(x_train, y_train, epochs=10, batch_size=32)

# Evalúa el modelo en el conjunto de test
_, accuracy = modelo_simple.evaluate(x_test, y_test, verbose=0)
print('Precisión del modelo simple en test:', accuracy)

Epoch 1/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2592 - loss: 2.1501   
Epoch 2/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6731 - loss: 1.3434 
Epoch 3/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8039 - loss: 0.8640 
Epoch 4/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8315 - loss: 0.6256 
Epoch 5/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8669 - loss: 0.5110 
Epoch 6/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9140 - loss: 0.3761 
Epoch 7/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9507 - loss: 0.3189 
Epoch 8/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9711 - loss: 0.2581 
Epoch 9/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9675 - loss: 0.2305 
Epoch 10/10
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9833 - loss: 0.1872 
Precisión del modelo simple en test: 0.868399977684021


### Entrenamiendo del modelo semisupervisado

El entrenamiento del sistema semisupervisado se realiza en dos pasos.

1. Se entrena el autoencoder con todos los datos (etiquetados y sin etiquetar).
1. Se entrena un clasificador simple (una o dos capas), teniendo como entrada la salida del encoder (**code**) de los datos etiquetados.

<font color='red'>NOTA:</font> para entrenar (y predecir) vamos a utilizar las funciones que hemos definido en el autoencoder y en el clasificador.

In [80]:
# TODO: implementa el algoritmo semisupervised_training.

def semisupervised_training(autoencoder, classifier, x_train, y_train, unlabeled_data):
  # 1. Entrena el autoencoder con todos los datos (etiquetados y sin etiquetar)
  autoencoder.fit(np.concatenate([x_train, unlabeled_data]))

  # 2. Entrena el clasificador con la salida del encoder de los datos etiquetados
  encoded_x_train = autoencoder.get_encoded_data(x_train)
  classifier.fit(encoded_x_train, y_train)

### Entrenamos nuestro modelo

Usa lo hecho anteriormente para entrenar tu clasificador de una manera semi-supervisada.

In [81]:
# Crea tu autoencoder y tu clasificador
autoencoder = MiAutoencoder(input_shape=(784,))  # Define la forma de entrada del autoencoder
classifier = MiClasificador()

In [82]:
# TODO: Entrena tu modelo
semisupervised_training(autoencoder, classifier, x_train, y_train, unlabeled_train)

Epoch 1/20
235/235 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - loss: 0.0992
Epoch 2/20
235/235 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 0.0341
Epoch 3/20
235/235 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 0.0256
Epoch 4/20
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 0.0217
Epoch 5/20
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: 0.0195
Epoch 6/20
235/235 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.0180
Epoch 7/20
235/235 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.0168
Epoch 8/20
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - loss: 0.0158
Epoch 9/20
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - loss: 0.0148
Epoch 10/20
235/235 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 0.0139
Epoch 11/20
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - loss: 0.0133
Epoch 12/20
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: 0.0129
Epoch 13/20
235/235 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.0124
Epoch 14/20
235/235 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - loss: 0.0121
Epoch 15/20
235/235 ━━━━━━━━━━━━━━━━━━━━ 3s

In [83]:
# TODO: Obtén la precisión sobre el conjunto de test
pred_data = autoencoder.get_encoded_data(x_test)
print('Test accuracy :', classifier.score(pred_data, y_test))

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Test accuracy : 0.7085000276565552


## Mejorando el código

nuestro modelo actual requiere de dos pasos para entrenarse, pero podría realizarse en un único paso si **creamos un modelo con las dos salidas (autoencoder y clasificador)**.

Para ello, hay que tener en cuenta que, en los datos sin etiquetar, su contribución al clasificador debería ser nula.


### TRABAJO: Crea el nuevo modelo y modifica la función semisupervised_training para tener en cuenta todos los puntos mencionados anteriormente

In [125]:
# TODO: crea el nuevo modelo

# TODO: crea tu propio clasificador

class MiClasificadorSemisupervisado:

    def __init__(self, input_shape):
        # TODO : define el modelo y compílalo
        input_layer = tf.keras.layers.Input(shape=input_shape)

        # Encoder
        x = tf.keras.layers.Dense(128, activation='relu')(input_layer)
        x = tf.keras.layers.Dense(64, activation='relu')(x)
        encoded = tf.keras.layers.Dense(32, activation='relu')(x)

        # Decoder
        x = tf.keras.layers.Dense(64, activation='relu')(encoded)
        x = tf.keras.layers.Dense(128, activation='relu')(x)
        decoded = tf.keras.layers.Dense(input_shape[0], activation='sigmoid')(x)

        # Clasificador
        classifier_output = tf.keras.layers.Dense(10, activation='softmax')(encoded)

        # Modelo completo
        self.model = tf.keras.models.Model(inputs=input_layer, outputs=[decoded, classifier_output])

        # Compilación
        self.model.compile(
        optimizer='adam',
        loss=['mse', 'sparse_categorical_crossentropy'],
        loss_weights=[0.5, 0.5],
        metrics=[None, 'accuracy']  # ✅ Accuracy solo en el clasificador
    )


    def fit(self, X, y, unlabeled_data):
        # TODO: entrena el modelo. Escoge el tamaño de batch y el número de epochs que quieras, y define bien el sample_weight

        # Etiquetas ficticias para los datos no etiquetados
        dummy_labels = np.zeros((unlabeled_data.shape[0],))

        # Combinación de datos
        all_data = np.concatenate([X, unlabeled_data])
        all_labels = np.concatenate([y, dummy_labels])

        # Pesos para las pérdidas
        ae_weights = np.ones(all_data.shape[0])
        clf_weights = np.concatenate([
            np.ones(X.shape[0]),
            np.zeros(unlabeled_data.shape[0])
        ])

        # Entrenamiento
        self.model.fit(
            all_data, [all_data, all_labels],
            sample_weight=[ae_weights, clf_weights],
            epochs=20,
            batch_size=256,
            verbose=0
        )

    def predict(self, X):
        # TODO: devuelve la clase ganadora del clasificador
        _, predictions = self.model.predict(X, verbose=0)
        return np.argmax(predictions, axis=-1)

    def predict_proba(self, X):
        # TODO: devuelve la probabilidad del clasificador
        _, predictions = self.model.predict(X, verbose=0)
        return predictions

    def score(self, X, y):
        results = self.model.evaluate(X, [X, y], verbose=0)
        clf_accuracy = results[-1]  # La última métrica es la accuracy del clasificador
        return clf_accuracy

    def __del__(self):
        # elimina todos los modelos que hayas creado
        tf.keras.backend.clear_session() # Necesario para liberar la memoria en GPU


In [126]:
# TODO: reescribe la función semisupervised_training para incorporar las mejoras mencionadas anteriormente

def semisupervised_training_v2(model, x_train, y_train, unlabeled_data):
    # Entrenamos el modelo usando los datos etiquetados y no etiquetados
    model.fit(x_train, y_train, unlabeled_data)

    # Evaluamos el modelo en el conjunto de test
    test_acc = model.score(x_test, y_test)

    # Mostramos el resultado
    print("Accuracy en test (modelo semisupervisado):", test_acc)


In [127]:
# TODO: Crea y entrena tu clasificador
model = MiClasificadorSemisupervisado(input_shape=(784,))
semisupervised_training_v2(model, x_train, y_train, unlabeled_train)

Accuracy en test (modelo semisupervisado): 0.8776999711990356


In [129]:
# TODO: Obtén la precisión sobre el conjunto de test
print('Test accuracy :', model.score(x_test, y_test))

Test accuracy : 0.8776999711990356


# Hay vida más allá del autoencoder

¿Has probado a utilizar otro método distinto del autoencoder para obtener una respresentación similar a la salida del encoder? La idea es la siguiente:

1. Define un modelo $model$ convolucional similar al encoder de un autoencoder (la entrada es el tamaño de la imagen, la salida el vector de representación)
1. Define una capa de salida $cluster$ que, partiendo de la salida de model, nos devuelva una salida con el mismo número de clases que el dataset a utilizar (la entrada es el vector de representación), usando softmax como activación de salida
1. Para cada batch de entrenamiento $X$:  # Usa un batch alto, mínimo 128
  1. Modifica las imágenes de entrada con [data_augmentation](https://www.tensorflow.org/tutorials/images/data_augmentation?hl=es-419), llámala $augX_1$.
  1. Modifica otra vez las imágenes de entrada con [data_augmentation_2](https://www.tensorflow.org/tutorials/images/data_augmentation?hl=es-419), llámala $augX_2$.
  1. $augX_{1comp} \leftarrow model(augX_1)$
  1. $augX_{2comp} \leftarrow model(augX_2)$
  1. $cX_{1comp} \leftarrow cluster(augX_{1comp})$
  1. $cX_{2comp} \leftarrow cluster(augX_{2comp})$
  1. $M \leftarrow augX_{1comp} ~ augX_{2comp}^T$
  1. $loss_C \leftarrow cX_{1comp}(1 - cX_{1comp}) + cX_{2comp}(1 - cX_{2comp})$ # Puede que tengas que crear tu [propia función de coste](https://keras.io/api/losses/#creating-custom-losses)
  1. $loss_M \leftarrow crossentropy(I, softmax(M/\tau, axis=1)))$ # Puede que tengas que crear tu [propia función de coste](https://keras.io/api/losses/#creating-custom-losses)
    1. $\tau$ es un hiperparámetro que se suele definir a 5.0
  1. $loss \leftarrow loss_M + \lambda~loss_C$
    1. $\lambda$ es un hiperparámetro (puedes probar con 0.5)


In [135]:
# Escribe aquí la solución. Crea tantos bloques de código como necesites. Puedes utilizar la siguiente red para generar distorsiones

data_augmentation = tf.keras.models.Sequential(
    [
        # tf.keras.layers.RandomFlip("horizontal"),  # Puede ser util en otros casos
        tf.keras.layers.RandomRotation(0.05),
        tf.keras.layers.RandomTranslation(0.15, 0.15),
        tf.keras.layers.RandomZoom(.15),
    ]
)

data_augmentation_2 = tf.keras.models.Sequential(
    [
        # tf.keras.layers.RandomFlip("horizontal"),  # Puede ser util en otros casos
        tf.keras.layers.RandomTranslation(0.15, 0.15),
        tf.keras.layers.Resizing(48, 48), # para CIFAR, para MNIST usar 40 en lugar de 48
        tf.keras.layers.RandomCrop(32, 32), # para CIFAR, para MNIST usar 28 en lugar de 32
    ]
)


# ¡ENHORABUENA! Has completado la práctica de modelos generativos.


# Trabajo extra

¿Has probado a hacer el autoencoder totalmente convolucional? Para el *decoder* puedes usar las funciones [UpSampling2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/UpSampling2D) o [Conv2DTranspose](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2DTranspose).